# Weigh a planet: first one you made, then a real one

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/04_weigh_a_planet.ipynb)

You fit an orbit to radial velocities twice. First to velocities you make yourself, where you know the answer, so you can see what a fit gets right and what it only appears to. Then to a real planet from the TESS-Keck Survey, whose answer is published.

Part 1 fits a planet you make yourself. Part 2 weighs a real planet, checks your answer against the paper, and puts it together with your radius from notebook 3. An optional Extra at the end explains a swing you will see in the worked example.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. At the end of each part, one cell puts all your answers into a single table and saves it with your plots. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None:
    %pip install -q "habtrain @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/04_weigh_a_planet.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## Part 1: Fit a planet you made yourself

Three terms, used from here on.

- **Posterior:** how likely each value of a parameter is, given the data and what you allowed beforehand (here, for example, that K cannot be negative). A fit gives you a shape, not a single number.
- **MCMC** (Markov chain Monte Carlo): a way of mapping that shape. Many walkers wander through the possible values and spend more time where the model fits the data better. This notebook uses emcee ([Foreman-Mackey et al. 2013](https://scixplorer.org/abs/2013PASP..125..306F/abstract)).
- **Corner plot:** every fitted parameter plotted against every other, plus each one on its own. Read it to see which parameters trade off against each other.

The model is a circular orbit: the star's velocity swings as a sine wave with amplitude K, the semi-amplitude from notebook 1, and the period P. The fit also finds an **offset**, the star's average velocity, and a **jitter**, extra scatter beyond the error bars that stands for everything the model leaves out.

> **Write down: the planet you make**
>
> - What wobble do you give your planet, in m/s? Start with something clear, such as `10`. Put it in `TRUE_K`.
> - What period, in days? Such as `7`. Put it in `TRUE_P`.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
TRUE_K = ...   # the wobble you give your planet, in m/s
TRUE_P = ...   # its period, in days

# If either is left empty, use 10 m/s and 7 days so the cells below still run.
K_true = ht.record.fill(TRUE_K, 10.0, "TRUE_K")
P_true = ht.record.fill(TRUE_P, 7.0, "TRUE_P")
ht.record.save("made", {"K (m/s)": TRUE_K, "P (days)": TRUE_P}, "the planet you made")

# Make 40 nights of velocities for that planet, with a spectrograph's noise added.
t, v, err = ht.fit.fake_velocities(K_true, P_true, n=40, error=2.0, jitter=2.0)

# Left: the velocities over time. Right: the same points folded on the period you chose.
fig = ht.plot.velocities(t + 2457000, v, err, P=P_true, T0=2457000, title="Your planet's velocities, 40 nights")
plt.show()

In [ ]:
# Fit a circular orbit: the period stays fixed, the fit finds K, the offset and the jitter.
fit_full = ht.fit.fit_circular(t, v, err, P=P_true, T0=0.0)

# The middle of the posterior for K, and how far its 68% range reaches below and above.
K, minus, plus = fit_full.value("K")
print(f"Fitted K = {K:.2f} -{minus:.2f} +{plus:.2f} m/s   (you made {K_true} m/s)")
print("Converged:", fit_full.converged)

# Every fitted parameter against every other; the blue lines mark the K you made.
fig_corner = ht.plot.corner(fit_full, truths={"K": K_true})
plt.show()

**Converged** means the walkers ran at least 50 times longer than it takes them to forget where they started (the autocorrelation time). If it says False, run the fit again with more steps, for example `ht.fit.fit_circular(t, v, err, P=P_true, T0=0.0, steps=10000)`.

The two numbers after K say how far the middle 68% of the posterior reaches below and above the middle value. The blue lines on the corner plot mark the value you made. A 68% range leaves out the true value about one time in three, so a value just outside it is normal, not a mistake.

> **Write down: what the fit gave back**
>
> - Is the K you made inside the fitted range? `"yes"` or `"no"`. Put it in `INSIDE`.
> - Report the result the way you would in a paper: K with its range, in one line. Put it in `REPORTED`.
>
> **Where to look:** the printed K and the corner plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
INSIDE = ...        # "yes" or "no"
REPORTED = ...      # one line, in quotes, for example "K = 10.8 (+0.5, -0.5) m/s"

# Keep the fit's answer and yours for the hand-in table.
ht.record.save("fitted", {"K (m/s)": round(K, 2), "minus": round(minus, 2), "plus": round(plus, 2)}, "the K your fit gave back")
ht.record.save("inside", INSIDE, "whether the K you made is inside the fitted range")
ht.record.save("reported", REPORTED, "your K, as you would report it in a paper")

**Predict first**

You are about to keep only 20 of your 40 nights, picked at random, and fit again. What happens to the fitted K, and to its width?

In [ ]:
# Your prediction: what happens to the fitted K, and to its width, with half the nights?
MY_PREDICTION = ...   # one line, in quotes
ht.record.predict("half", MY_PREDICTION)

In [ ]:
# This only runs once you have made your prediction above.
if ht.record.revealed("half"):
    # Keep 20 of the 40 nights, picked at random, and fit them the same way.
    keep = np.sort(np.random.default_rng(2).choice(len(t), 20, replace=False))
    t2, v2, err2 = t[keep], v[keep], err[keep]
    fit_half = ht.fit.fit_circular(t2, v2, err2, P=P_true, T0=0.0)
    Kh, mh, ph = fit_half.value("K")
    print(f"40 nights: K = {K:.2f} -{minus:.2f} +{plus:.2f} m/s")
    print(f"20 nights: K = {Kh:.2f} -{mh:.2f} +{ph:.2f} m/s")

Now make the planet ten times lighter, so its wobble is a tenth as big, and fit the full 40 nights again.

**What you need**

The survey in Part 2 confirmed a new planet by its mass only when the fit put the mass at least **3 sigma** away from zero: the middle of the posterior at least three times its uncertainty, where the uncertainty is half the width of its 68% range (the average of the minus and plus numbers). Below that it still reports the mass, as a weak measurement, and gives only an upper limit when there is barely any signal. Its stated goal was 5 sigma, which it reached for 58 of the 126 planets and candidate signals it fitted ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786)).

In [ ]:
# A planet ten times lighter, so its wobble is a tenth as big, on 40 nights.
t3, v3, err3 = ht.fit.fake_velocities(K_true / 10, P_true, n=40, error=2.0, jitter=2.0)
fit_light = ht.fit.fit_circular(t3, v3, err3, P=P_true, T0=0.0)

# The middle divided by its uncertainty (half the 68% range): 3 or more is the survey's bar for a confirmed mass.
Kl, ml, pl = fit_light.value("K")
print(f"Made: K = {K_true / 10} m/s.   Fitted: K = {Kl:.2f} -{ml:.2f} +{pl:.2f} m/s   "
      f"(middle divided by uncertainty: {Kl / ((ml + pl) / 2):.1f})")

Compare the K you made with the one the fit found. When a signal is small next to the scatter, the fitted value wanders, and because K cannot be negative it more often comes out too big than too small.

> **Write down: what changes when the data gets worse**
>
> - What changed when you used half the nights, and did it match your prediction? Put it in `HALF_DATA`.
> - For the planet ten times lighter: would that count as a confirmed mass? Why? Put it in `LIGHTER`.
>
> **Where to look:** the three fits above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
HALF_DATA = ...   # one line, in quotes
LIGHTER = ...     # one line, in quotes

# Keep the answers for your hand-in table.
ht.record.save("half_data", HALF_DATA, "what changed with half the nights")
ht.record.save("lighter", LIGHTER, "whether the lighter planet counts as a confirmed mass")

### Hand in Part 1

Run the cell below. It shows all your answers from Part 1 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `fake-planet-fitted`, and download them.

In [ ]:
# Your Part 1 answers and corner plot, saved as your hand-in.
ht.record.handin("fake-planet-fitted", ["made", "fitted", "inside", "reported", "half_data", "lighter"],
                 figures={"corner": fig_corner})

## Part 2: Weigh a real planet, then check your answer against the paper

The TESS-Keck Survey spent three years measuring the masses of planets that TESS found by their transits, with two spectrographs: HIRES on the Keck I telescope and the Levy spectrograph on the Automated Planet Finder (APF). Its final analysis released all of it: 9,204 measurements of 86 stars, 9,000 of them with a velocity (the rest carry only an S-value, the activity measure from notebook 2) ([Polanski et al. 2024](https://arxiv.org/abs/2405.14786)).

One of the survey's planets was confirmed by this lab, which also found its outer companion. [Hill et al. 2024](https://arxiv.org/abs/2402.03498) reports TOI-1386 b, a warm sub-Saturn that transits, and TOI-1386 c, a planet at least as heavy as Saturn on a 228-day orbit that does not transit. Same survey and spectrograph, and a fuller version of the fit you are about to run. If you want to see what the finished work looks like, its radial velocity section is a short read before you start.

The model is the one from Part 1, with three changes for real data:
- The period and the transit time come from the TESS transits and stay fixed. The fit finds K.
- Each spectrograph gets its own offset and its own jitter. The two do not measure the same absolute velocity, and a fit that forces one offset on both data sets gets K wrong.
- It has no term for the star's own activity. The survey modelled activity for a few stars, so for those this fit will not match the paper as closely.

In [ ]:
# Every velocity the survey released: star, time, velocity, its error, S-value and telescope.
rvs = ht.data.load_rvs()

# The planets you can fit with this model (one planet, circular orbit), with what is known before fitting.
planets = ht.data.planets()
print(len(rvs), "velocities;", len(planets), "planets you can weigh with this model")

### Worked example: TOI-1181 b

A hot Jupiter (a giant planet on an orbit so close to its star that it is very hot) with a two-day orbit and a wobble you cannot miss. This works through every step once; then you repeat them on your own planet.

In [ ]:
TKS_CREDIT = "Data: TESS-Keck Survey release, Polanski et al. 2024, VizieR J/ApJS/272/32."
# TOI-1181 b's published period and transit time, and its star's velocities.
orb = ht.data.orbit("TOI-1181 b")
d = rvs[rvs.TKS == orb["TKS"]]
print(f"{len(d)} velocities; period {orb['P']:.4f} days, fixed from the transits")

# Which periods repeat in the velocities? The dashed line marks the transit period.
periods, power = ht.fit.periodogram(d.BJD, d.RV, 0.5, 50)
fig = ht.plot.periodogram(periods, power, "days", "TOI-1181: which periods repeat in the velocities?", mark=orb["P"], credit=TKS_CREDIT)
plt.show()

# Fit every velocity: K, plus one offset and one jitter per telescope.
fit_all = ht.fit.fit_circular(d.BJD, d.RV, d.eRV, orb["P"], orb["T0"], d.Inst)
K_all = fit_all.value("K")[0]
print(f"Every velocity: K = {K_all:.1f} m/s, jitter {fit_all.value('jitter HIRES')[0]:.1f} m/s")
# The velocities over time, and folded on the period with the fitted orbit drawn through them.
fig = ht.plot.velocities(d.BJD, d.RV, d.eRV, d.Inst, P=orb["P"], T0=orb["T0"], K=K_all,
                         offsets={"HIRES": fit_all.value("offset HIRES")[0]}, title="TOI-1181 b, every velocity", credit=TKS_CREDIT)
plt.show()

The periodogram's tallest peak sits on the dashed line at the transit period, so the velocities and the transits agree. But look at the folded plot near phase 0, the middle of the transit (phase runs from -0.5 to 0.5 over one orbit): a crowd of points there does not follow the curve.

Those points come from one night. On 8 September 2020, 40 of these 87 velocities were taken one after another, before, during and after a transit, when the planet crossed in front of its star. The star spins, so one half of it is coming towards you and the other half moving away. While the planet covers part of one half, the light from that half is missing, and the star's measured velocity shifts for a few hours. This is the Rossiter-McLaughlin effect; the Extra at the end of this notebook looks at it closely. This model has no way to describe it, so it does the next best thing: it inflates the jitter, the scatter it cannot explain, to about 27 m/s.

The fix is to leave out the hours of the transit and fit again.

In [ ]:
# Which velocities fall inside a transit (plus a quarter of an hour either side)?
hours = ht.data.transit_duration("TOI-1181 b")
during = ht.fit.in_transit(d.BJD, orb["P"], orb["T0"], hours)

# Leave those out, and fit again.
out = d[~during]
fit_ex = ht.fit.fit_circular(out.BJD, out.RV, out.eRV, orb["P"], orb["T0"], out.Inst)
K_ex, m_ex, p_ex = fit_ex.value("K")
print(f"Transit lasts {hours:.1f} hours; {during.sum()} velocities fall inside it and are left out.")
_, m_all, p_all = fit_all.value("K")
print(f"Every velocity:        K = {K_all:.1f} -{m_all:.1f} +{p_all:.1f} m/s, jitter {fit_all.value('jitter HIRES')[0]:.1f} m/s")
print(f"Transit hours removed: K = {K_ex:.1f} -{m_ex:.1f} +{p_ex:.1f} m/s, jitter {fit_ex.value('jitter HIRES')[0]:.1f} m/s.  Converged: {fit_ex.converged}")
fig = ht.plot.velocities(out.BJD, out.RV, out.eRV, out.Inst, P=orb["P"], T0=orb["T0"], K=K_ex,
                         offsets={"HIRES": fit_ex.value("offset HIRES")[0]}, title="TOI-1181 b, transit hours left out", credit=TKS_CREDIT)
plt.show()

# From K to mass, using the star's mass from the NASA Exoplanet Archive.
star = ht.data.star_mass("TOI-1181 b")
mass_ex = ht.fit.mass_from_wobble(K_ex, orb["P"], star)
mass_ex_err = ht.fit.mass_from_wobble((m_ex + p_ex) / 2, orb["P"], star)
print(f"Star: {star} solar masses.  Planet: {mass_ex:.0f} ± {mass_ex_err:.0f} Earth masses, about {mass_ex / 317.8:.1f} Jupiter masses")

# Check against the paper: TOI-1181 b's published mass is 377 ± 17 Earth masses (Polanski et al. 2024, table 4).
apart = ht.fit.sigma_apart(mass_ex, mass_ex_err, 377, 17)
print(f"Published: 377 ± 17 Earth masses. Yours is {apart:.1f} sigma from it.")

With the transit hours left out, the jitter falls to about a third, the uncertainty on K shrinks, and the mass lands within 1 sigma of the published one. **Sigma apart** is the difference between two results divided by their combined uncertainty. You fit the same velocities as the paper with nearly the same model, so under 1 sigma is a match. Over 1, check these in order: the fit had not converged; velocities from inside a transit were left in; the two spectrographs were given one offset; or the survey modelled the star's activity, which this fit does not. The first fit was not obviously broken: it converged and its curve went through the points. The high jitter is what showed something was wrong.

The star's mass comes from the NASA Exoplanet Archive. Because the planet transits, its orbit is seen almost edge on, so K gives the mass itself rather than a lower limit.

### Your planet

Weigh the planet you measured in notebook 3, so you can put its radius and mass together at the end. Choose it in the menu under the map. "Rock or gas?" at the end needs your radius from notebook 3; if you skipped it, run notebook 3 for this planet first.

It is normal for a first fit not to converge. If `Converged` says False, run the fit again with `steps=10000`.

In [ ]:
# The planets you can weigh, numbered, and a menu that circles your choice on the map.
# TOI-1181 b, the worked example, is left out.
planets = planets[planets.planet != "TOI-1181 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()
chooser = ht.choose.planet(planets, ht.plot.planet_map)

> **Write down: your planet's velocities**
>
> - Choose your planet in the menu above. If the menu does not appear, put its number from the map in `MY_PICK`.
> - Keep only your star's velocities from `rvs`. Put the code in `mine`.
>
> **Where to look:** the map and the menu above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

`orb["TKS"]` is the id of your star in the survey.

</details>

<details><summary>Stuck? Open hint 2</summary>

`rvs[rvs.TKS == orb["TKS"]]`

</details>

In [ ]:
MY_PICK = ...   # only if the menu did not appear: your planet's number on the map

# Your planet: what you chose in the menu, else the number you typed, else a random one.
planet = ht.choose.resolve(chooser, MY_PICK, planets, "planet", "planet")
ht.record.save("my_planet", planet, "your planet")

# Its published period and transit time, which stay fixed in the fit.
orb = ht.data.orbit(planet)

mine = ...   # your code: only your star's velocities
mine = ht.record.fill(mine, rvs[rvs.TKS == orb["TKS"]], "mine")
print(f"{len(mine)} velocities from {', '.join(sorted(mine.Inst.unique()))}; period {orb['P']:.4f} days")

# How many of them were taken during a transit? If any were, leave them out as the worked example did.
during_mine = ht.fit.in_transit(mine.BJD, orb["P"], orb["T0"], ht.data.transit_duration(planet))
print(f"{during_mine.sum()} of them were taken during a transit.")

# Your star's velocities over time, coloured by telescope.
fig = ht.plot.velocities(mine.BJD, mine.RV, mine.eRV, mine.Inst, title=planet, credit=TKS_CREDIT)
plt.show()

**Before you fit: is it the planet or the star?** Run the two periodograms below, one of the velocities and one of the S-values from the same spectra. As notebook 2 showed, a star's activity cycle can look like a planet.

In [ ]:
# Each spectrograph has its own zero point, so take each one's median off before looking for periods.
rv_centred = mine.RV - mine.groupby("Inst").RV.transform("median")
s_centred = mine.SHK - mine.groupby("Inst").SHK.transform("median")

# Periodogram of the velocities: does the tallest peak sit on the transit period (dashed)?
p1, w1 = ht.fit.periodogram(mine.BJD, rv_centred, 0.5, 100)
fig = ht.plot.periodogram(p1, w1, "days", f"{planet}: velocities", mark=orb["P"], credit=TKS_CREDIT)
plt.show()
# Periodogram of the activity (S-values from the same spectra): does the star repeat at that period too?
ok = s_centred.notna()
p2, w2 = ht.fit.periodogram(mine.BJD[ok], s_centred[ok], 0.5, 100)
fig = ht.plot.periodogram(p2, w2, "days", f"{planet}: activity (S-values)", mark=orb["P"], credit=TKS_CREDIT)
plt.show()

> **Write down: planet or star**
>
> - Does the velocity peak sit on the transit period, and does the activity show the same period? What do you conclude? One or two lines. Put it in `PLANET_OR_STAR`.
>
> **Where to look:** the two periodograms, at the dashed line.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
PLANET_OR_STAR = ...   # one or two lines, in quotes

# Keep the answer for your hand-in table.
ht.record.save("planet_or_star", PLANET_OR_STAR, "whether the signal is the planet or the star")

> **Write down: your planet's wobble and mass**
>
> - Fit your star's velocities, with each spectrograph separate. Put the code in `my_fit`. The cell then prints K, whether the fit converged, and your planet's mass, and keeps them for your hand-in.
>
> **Where to look:** the printed K, the corner plot and the folded plot.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

<details><summary>Stuck? Open hint 1</summary>

Copy the worked example: `ht.fit.fit_circular(mine.BJD, mine.RV, mine.eRV, orb["P"], orb["T0"], mine.Inst)`.

</details>

<details><summary>Stuck? Open hint 2</summary>

If any of your velocities fall inside a transit, leave them out as the worked example did: use `mine[~during_mine]` in place of `mine`.

</details>

In [ ]:
my_fit = ...   # your code: the fit of your star's velocities, each spectrograph separate

if ht.record.filled(my_fit):
    # K and its 68% range, then the mass from K and the star's mass, with the mass's uncertainty.
    K, minus, plus = my_fit.value("K")
    star = ht.data.star_mass(planet)
    my_mass = ht.fit.mass_from_wobble(K, orb["P"], star)
    my_mass_err = ht.fit.mass_from_wobble((minus + plus) / 2, orb["P"], star)
    print(f"K = {K:.2f} -{minus:.2f} +{plus:.2f} m/s, which is {K / ((minus + plus) / 2):.1f} sigma from zero.  Converged: {my_fit.converged}")
    print(f"Star {star} solar masses.  Mass = {my_mass:.1f} ± {my_mass_err:.1f} Earth masses "
          f"(the ± is from K only; the star's own mass uncertainty adds more)")
    if K / ((minus + plus) / 2) < 3:
        print("Under 3 sigma: treat this mass as weak, close to an upper limit, and say so in ROCK_OR_GAS.")
    # The corner plot, then the velocities folded with your fitted orbit through them.
    my_corner = ht.plot.corner(my_fit, credit=TKS_CREDIT)
    plt.show()
    offsets = {n: my_fit.value(f"offset {n}")[0] for n in my_fit.telescopes}
    my_folded = ht.plot.velocities(mine.BJD, mine.RV, mine.eRV, mine.Inst, P=orb["P"], T0=orb["T0"],
                                   K=K, offsets=offsets, title=planet, credit=TKS_CREDIT)
    plt.show()
    # Keep your results for the hand-in table.
    ht.record.save("wobble", {"K (m/s)": round(K, 2), "minus": round(minus, 2), "plus": round(plus, 2)}, "your fitted K")
    ht.record.save("mass", {"mass (Earth masses)": round(my_mass, 1), "uncertainty": round(my_mass_err, 1)}, "your planet's mass")
    ht.record.save("converged", str(my_fit.converged), "whether your fit converged")

### Rock or gas?

You now have both halves: the radius from the dip in notebook 3, and the mass from the wobble here.

**What you need**

Density alone does not settle it, because rock compresses: the bigger a rocky planet is, the denser it gets, so a big rocky planet can be much denser than Earth. The better test is to ask what radius rock would give a planet of your mass. For a planet made of rock and iron, [Zeng, Sasselov and Jacobsen 2016](https://arxiv.org/abs/1512.08827) give

radius = (1.07 - 0.21 × core fraction) × mass<sup>1/3.7</sup>, in Earth units, where the core fraction is the share of the planet's mass in its iron core,

fitted for about 1 to 8 Earth masses. Earth's core fraction is about a third (0.325). A core fraction of 0, no iron at all, gives the biggest a rocky planet of that mass can be. **If your planet is bigger than that, rock alone cannot make it: it needs something lighter around it, such as gas or water.** Above 8 Earth masses, treat the line as a rough guide. Your radius from notebook 3 usually runs a little high, and density goes as radius cubed, so a density worked out from it runs low.

The lab's 0.8 Earth radii line, from notebook 1, is about rocky planets at Earth's distance from a sun-like star.

> **Write down: what your planet is made of**
>
> - What radius did you measure in notebook 3, in Earth radii? Put it in `MY_RADIUS`. (Use the planet you weighed here.)
> - The cell then plots your planet against what rock could make, and prints the biggest rocky radius for your mass.
> - In one or two lines: is your planet rock, does it need something lighter, or is it too close to the line to say? Could it ever test the lab's line? Put it in `ROCK_OR_GAS`.
>
> **Where to look:** the plot the cell draws, and your mass printed above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
MY_RADIUS = ...     # your radius from notebook 3, in Earth radii
ROCK_OR_GAS = ...   # one or two lines, in quotes

# This needs your own fit from above (my_mass) and your radius from notebook 3.
if ht.record.filled(MY_RADIUS) and "my_mass" in dir():
    # The biggest a rocky planet of your mass can be (no iron), and an Earth-like mix, for comparison.
    biggest_rock = float(ht.fit.rocky_radius(my_mass, 0.0))
    earth_like = float(ht.fit.rocky_radius(my_mass, 0.325))
    density = my_mass / MY_RADIUS ** 3 * 5.514          # grams per cubic centimetre
    print(f"Mass {my_mass:.1f} Earth masses, radius {MY_RADIUS} Earth radii, density {density:.2f} g/cm3")
    print(f"Rock alone would give at most {biggest_rock:.2f} Earth radii at this mass (Earth-like mix: {earth_like:.2f})")
    fig_rock = ht.plot.mass_radius(my_mass, MY_RADIUS, planet, mass_err=my_mass_err)
    plt.show()
    ht.record.save("density", round(density, 2), "your planet's density (g/cm3)")
    ht.record.save("biggest_rock", round(biggest_rock, 2), "the biggest rocky radius for your mass (Earth radii)")
elif "my_mass" not in dir():
    print("Run your own fit above first: this needs your planet's mass.")
ht.record.save("rock_or_gas", ROCK_OR_GAS, "rock or something lighter, and whether it could test the lab's line")

### Hand in Part 2

Run the cell below. It shows all your answers from Part 2 in one table. Check them, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `real-planet-fitted`, and download them.

Once you have handed in, the step shows the answer key: the published mass of every planet on the list, so you can see how close you got.

In [ ]:
# Your Part 2 answers and plots, saved as your hand-in.
figs = {}
if "my_corner" in dir():
    figs = {"corner": my_corner, "folded": my_folded}
if "fig_rock" in dir():
    figs["rock_or_gas"] = fig_rock
ht.record.handin("real-planet-fitted",
                 ["my_planet", "planet_or_star", "wobble", "mass", "converged", "density", "biggest_rock", "rock_or_gas"], figures=figs)

## Extra: why does the star's velocity jump during a transit?

Optional. This picks up the crowd of TOI-1181 b's velocities near phase 0 that did not follow the orbit in the worked example. The first plot circles them.

**What you need**

A star spins, so one half of it is turning towards you and the other half away. Light from the half coming towards you is shifted slightly blue, light from the other half slightly red, and the two balance out. When a planet crosses in front, it blocks some light from one half first and then from the other. While it blocks part of the half turning towards you, the star looks slightly redshifted and its measured velocity goes up. While it blocks part of the half moving away, the velocity goes down. This is the Rossiter-McLaughlin effect. A planet that orbits in the same direction its star spins crosses the half turning towards you first.

The order of the swing tells you which way the planet goes round compared with the star's spin.

In [ ]:
from matplotlib.patches import Ellipse

# This Extra runs on its own: it loads what it needs even if you skipped Part 2.
rvs = ht.data.load_rvs()
TKS_CREDIT = "Data: TESS-Keck Survey release, Polanski et al. 2024, VizieR J/ApJS/272/32."

# TOI-1181 b again: its orbit, its velocities, and how long each transit lasts.
orb_x = ht.data.orbit("TOI-1181 b")
dx = rvs[rvs.TKS == orb_x["TKS"]]
hours_x = ht.data.transit_duration("TOI-1181 b")

# Fit the orbit without the transit hours, as in the worked example.
outside = ~ht.fit.in_transit(dx.BJD, orb_x["P"], orb_x["T0"], hours_x)
fit_x = ht.fit.fit_circular(dx.BJD[outside], dx.RV[outside], dx.eRV[outside], orb_x["P"], orb_x["T0"], dx.Inst[outside])

# First, the folded plot from the worked example, with every velocity, and the swing circled.
fig_fold = ht.plot.velocities(dx.BJD, dx.RV, dx.eRV, dx.Inst, P=orb_x["P"], T0=orb_x["T0"], K=fit_x.value("K")[0],
                              offsets={"HIRES": fit_x.value("offset HIRES")[0]}, title="TOI-1181 b, every velocity",
                              credit=TKS_CREDIT)
ax_fold = fig_fold.axes[1]                         # the right-hand, folded panel
width = hours_x / 24 / orb_x["P"] * 1.6            # the transit's width in orbital phase, with room either side
ax_fold.add_patch(Ellipse((0, 0), width=width, height=220, fill=False, edgecolor="#d1495b", lw=2))
ax_fold.annotate("this swing", (width / 2, 90), xytext=(0.12, 125), color="#d1495b", fontsize=10,
                 arrowprops=dict(arrowstyle="->", color="#d1495b"))
plt.show()
# Now zoom in on that one night: 8 September 2020, when the swing was observed.
night = dx[np.floor(dx.BJD + 0.5) == 2459101]

# Hours from mid-transit, and what is left after taking the orbit out of each velocity.
h = ht.fit.hours_from_transit(night.BJD, orb_x["P"], orb_x["T0"])
left_over = night.RV - (ht.fit.circular(night.BJD, fit_x.value("K")[0], orb_x["P"], orb_x["T0"]) + fit_x.value("offset HIRES")[0])

# Plot what is left against hours from mid-transit, with the transit shaded.
fig_swing, ax = plt.subplots(figsize=(9, 3.8), dpi=110)
ax.axvspan(-hours_x / 2, hours_x / 2, color="#f2e8cf", label="during the transit")
ax.axhline(0, color="#999", lw=1)
ax.errorbar(h, left_over, night.eRV, fmt=".", color="#3b6ea5")
ax.set_xlabel("Hours from mid-transit"); ax.set_ylabel("Velocity minus the orbit (m/s)")
ax.set_title("TOI-1181 b, the night of 8 September 2020", fontsize=11); ax.legend(fontsize=8)
fig_swing.text(0.01, 0.005, TKS_CREDIT, fontsize=7, color="#666")
plt.show()

> **Write down: what the swing tells you**
>
> - During the transit, which comes first: an upward swing or a downward one? `"up first"`, `"down first"` or `"unsure"`. Put it in `FIRST_SWING`.
> - So does the planet go round in the same direction the star spins, or the opposite way? `"same"`, `"opposite"` or `"unsure"`, and why in a few words. Put it in `DIRECTION`.
>
> **Where to look:** the shaded part of the plot above.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
FIRST_SWING = ...   # "up first", "down first" or "unsure"
DIRECTION = ...     # "same", "opposite" or "unsure", and why

# Keep the answers for your hand-in table.
ht.record.save("first_swing", FIRST_SWING, "which swing comes first during the transit")
ht.record.save("direction", DIRECTION, "which way the planet goes round compared with the star's spin")

### Hand in the Extra

Run the cell below. It shows your two answers in one table. Hand in the Extra step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `transit-swing-read`, and download them.

Once you have handed in, the step shows the answer key: what the published measurement found.

In [ ]:
# Your Extra answers and the swing plot, saved as your hand-in.
ht.record.handin("transit-swing-read", ["first_swing", "direction"], figures={"swing": fig_swing})